In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
BASE = "/content/drive/MyDrive/AquaStrat"
!find "{BASE}" -maxdepth 4

/content/drive/MyDrive/AquaStrat
/content/drive/MyDrive/AquaStrat/coral_dataset
/content/drive/MyDrive/AquaStrat/coral_dataset/boundary
/content/drive/MyDrive/AquaStrat/coral_dataset/boundary/boundary.geojson
/content/drive/MyDrive/AquaStrat/coral_dataset/stats
/content/drive/MyDrive/AquaStrat/coral_dataset/stats/statistics.csv
/content/drive/MyDrive/AquaStrat/coral_dataset/1-License-and-Documentation
/content/drive/MyDrive/AquaStrat/coral_dataset/1-License-and-Documentation/1-Data-Licenses-and-Citation-Guidance.pdf
/content/drive/MyDrive/AquaStrat/coral_dataset/1-License-and-Documentation/Methods-Bathymetry.pdf
/content/drive/MyDrive/AquaStrat/coral_dataset/1-License-and-Documentation/Class-Descriptions-Geomorphic-Maps-v3.pdf
/content/drive/MyDrive/AquaStrat/coral_dataset/1-License-and-Documentation/2-Data-Attribution.csv
/content/drive/MyDrive/AquaStrat/coral_dataset/1-License-and-Documentation/Methods-Turbidity.pdf
/content/drive/MyDrive/AquaStrat/coral_dataset/1-License-and-Documen

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd, numpy as np, geopandas as gpd, xarray as xr
from shapely.geometry import box
import warnings
warnings.filterwarnings("ignore")

BASE = "/content/drive/MyDrive/AquaStrat"

# ---- grid ----
def build_equal_area_grid(region_code, lat_min, lat_max, lon_min, lon_max, cell_km=25):
    rows, grid_num, lat = [], 0, lat_min
    while lat < lat_max:
        dlat = cell_km / 111.0
        lon = lon_min
        km_per_lon = 111.320 * np.cos(np.radians(lat + dlat/2))
        dlon = cell_km / km_per_lon
        while lon < lon_max:
            rows.append({
                "grid_id": f"{region_code}_{grid_num:05d}", "region": region_code,
                "lat_min": lat, "lat_max": min(lat+dlat, lat_max),
                "lon_min": lon, "lon_max": min(lon+dlon, lon_max),
                "lat_center": lat+dlat/2, "lon_center": lon+dlon/2,
            })
            grid_num += 1
            lon += dlon
        lat += dlat
    return pd.DataFrame(rows)

grid_as = build_equal_area_grid("AS", 8, 24, 66, 76, cell_km=25)
grid_as_gdf = gpd.GeoDataFrame(
    grid_as,
    geometry=[box(r.lon_min, r.lat_min, r.lon_max, r.lat_max) for r in grid_as.itertuples()],
    crs="EPSG:4326"
)

# ---- land mask ----
world = gpd.read_file("https://naciscdn.org/naturalearth/110m/cultural/ne_110m_admin_0_countries.zip").to_crs("EPSG:4326")
centroids = gpd.GeoDataFrame(geometry=grid_as_gdf.geometry.centroid, crs="EPSG:4326")
on_land = gpd.sjoin(centroids, world[["geometry"]], how="left", predicate="within")
grid_as_gdf["is_land"] = ~on_land["index_right"].isna()
grid_as_gdf = grid_as_gdf[~grid_as_gdf["is_land"]].reset_index(drop=True)
print("Ocean cells:", grid_as_gdf.shape)

# ---- wildlife ----
wildlife = pd.read_csv(f"{BASE}/Wildlife.tsv", sep='\t', low_memory=False)
wildlife = wildlife[['decimalLatitude','decimalLongitude']].dropna()
wildlife_gdf = gpd.GeoDataFrame(wildlife, geometry=gpd.points_from_xy(wildlife.decimalLongitude, wildlife.decimalLatitude), crs="EPSG:4326")
joined = gpd.sjoin(wildlife_gdf, grid_as_gdf[["grid_id","geometry"]], how="inner", predicate="within")
counts = joined.groupby("grid_id").size().rename("wildlife_count")
grid_as_gdf = grid_as_gdf.merge(counts, on="grid_id", how="left").fillna({"wildlife_count": 0})
grid_as_gdf["wildlife_exposure"] = np.log1p(grid_as_gdf["wildlife_count"]) / np.log1p(grid_as_gdf["wildlife_count"]).max()
print("Wildlife attached:", grid_as_gdf["wildlife_count"].describe())

# ---- coral ----
coral = gpd.read_file(f"{BASE}/coral_dataset/Reef-Extent/reefextent.geojson")
joined_coral = gpd.sjoin(grid_as_gdf[["grid_id","geometry"]], coral, how="inner", predicate="intersects")
grid_as_gdf["habitat_sensitivity"] = grid_as_gdf["grid_id"].isin(joined_coral["grid_id"].unique()).astype(float)
print("Reef cells:", grid_as_gdf["habitat_sensitivity"].sum())

# ---- save checkpoint to Drive (survives runtime resets) ----
grid_as_gdf.drop(columns="geometry").to_csv(f"{BASE}/grid_as_progress.csv", index=False)
print("Saved checkpoint to Drive.")
grid_as_gdf.head()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Ocean cells: (2272, 10)
Wildlife attached: count    2272.000000
mean       37.064261
std       266.625025
min         0.000000
25%         0.000000
50%         0.000000
75%         0.000000
max      9402.000000
Name: wildlife_count, dtype: float64
Reef cells: 28.0
Saved checkpoint to Drive.


,grid_id,region,lat_min,lat_max,lon_min,lon_max,lat_center,lon_center,geometry,is_land,wildlife_count,wildlife_exposure,habitat_sensitivity
0,AS_00000,AS,8.0,8.225225,66.000000,66.226848,8.112613,66.113424,"POLYGON ((66.22685 8, 66.22685 8.22523, 66 8.2...",False,0.0,0.0,0.0
1,AS_00001,AS,8.0,8.225225,66.226848,66.453696,8.112613,66.340272,"POLYGON ((66.4537 8, 66.4537 8.22523, 66.22685...",False,0.0,0.0,0.0
2,AS_00002,AS,8.0,8.225225,66.453696,66.680544,8.112613,66.567120,"POLYGON ((66.68054 8, 66.68054 8.22523, 66.453...",False,0.0,0.0,0.0
3,AS_00003,AS,8.0,8.225225,66.680544,66.907392,8.112613,66.793968,"POLYGON ((66.90739 8, 66.90739 8.22523, 66.680...",False,0.0,0.0,0.0
4,AS_00004,AS,8.0,8.225225,66.907392,67.134240,8.112613,67.020816,"POLYGON ((67.13424 8, 67.13424 8.22523, 66.907...",False,0.0,0.0,0.0


In [ ]:
ds = xr.open_dataset(f"{BASE}/ocean_physics.nc")
print(ds)

<xarray.Dataset> Size: 184MB
Dimensions:    (time: 366, depth: 1, latitude: 175, longitude: 144)
Coordinates:
  * time       (time) datetime64[ns] 3kB 2025-01-01 2025-01-02 ... 2026-01-01
  * depth      (depth) float32 4B 0.494
  * latitude   (latitude) float32 700B 8.0 8.083 8.167 8.25 ... 22.33 22.42 22.5
  * longitude  (longitude) float32 576B 68.0 68.08 68.17 ... 79.75 79.83 79.92
Data variables:
    thetao     (time, depth, latitude, longitude) float32 37MB ...
    so         (time, depth, latitude, longitude) float32 37MB ...
    uo         (time, depth, latitude, longitude) float32 37MB ...
    vo         (time, depth, latitude, longitude) float32 37MB ...
    zos        (time, latitude, longitude) float32 37MB ...
Attributes:
    Conventions:       CF-1.11
    title:             daily mean fields from Global Ocean Physics Analysis a...
    institution:       MERCATOR OCEAN
    source:            MERCATOR GLORYS12V1
    history:           2023/06/01 16:20:05 MERCATOR OCEAN Netcd

In [ ]:
# annual mean across all days, drop the trivial single-depth dimension
ds_mean = ds.mean(dim="time", skipna=True).squeeze("depth")

def get_cell_ocean_stats(row, ds_mean):
    sub = ds_mean.sel(latitude=slice(row.lat_min, row.lat_max), longitude=slice(row.lon_min, row.lon_max))
    if sub.latitude.size == 0 or sub.longitude.size == 0:
        return pd.Series([np.nan]*5)
    return pd.Series([
        float(sub['thetao'].mean(skipna=True)),
        float(sub['so'].mean(skipna=True)),
        float(sub['uo'].mean(skipna=True)),
        float(sub['vo'].mean(skipna=True)),
        float(sub['zos'].mean(skipna=True)),
    ])

grid_as_gdf[['sst','salinity','current_u','current_v','ssh']] = grid_as_gdf.apply(
    lambda r: get_cell_ocean_stats(r, ds_mean), axis=1
)

print("Cells with valid ocean data:", grid_as_gdf['sst'].notna().sum(), "out of", grid_as_gdf.shape[0])
print(grid_as_gdf[['sst','salinity','current_u','current_v','ssh']].describe())

# checkpoint to Drive
grid_as_gdf.drop(columns="geometry").to_csv(f"{BASE}/grid_as_progress.csv", index=False)
print("Saved.")

Cells with valid ocean data: 1673 out of 2272
               sst     salinity    current_u    current_v          ssh
count  1673.000000  1673.000000  1673.000000  1673.000000  1673.000000
mean     28.898527    35.338920     0.020103    -0.012053     0.458630
std       0.519900     0.753747     0.037970     0.044930     0.044321
min      27.000763    30.024895    -0.113553    -0.214283     0.356000
25%      28.575563    34.795067    -0.002999    -0.034548     0.426230
50%      29.064131    35.518341     0.018900    -0.006842     0.464502
75%      29.332527    35.935051     0.040823     0.014220     0.490108
max      29.549200    36.309792     0.197455     0.164532     0.544511
Saved.


In [ ]:
grid_as_gdf["has_ocean_physics"] = grid_as_gdf["sst"].notna()
print(grid_as_gdf["has_ocean_physics"].value_counts())

# final checkpoint for AS
grid_as_gdf.drop(columns="geometry").to_csv(f"{BASE}/grid_as_final.csv", index=False)
print("AS grid finalized and saved.")

has_ocean_physics
True     1673
False     599
Name: count, dtype: int64
AS grid finalized and saved.


In [ ]:
grid_as_final = pd.read_csv(f"{BASE}/grid_as_final.csv")
print("Shape:", grid_as_final.shape)
print("Columns:", grid_as_final.columns.tolist())
print()
print(grid_as_final.describe())

Shape: (2272, 18)
Columns: ['grid_id', 'region', 'lat_min', 'lat_max', 'lon_min', 'lon_max', 'lat_center', 'lon_center', 'is_land', 'wildlife_count', 'wildlife_exposure', 'habitat_sensitivity', 'sst', 'salinity', 'current_u', 'current_v', 'ssh', 'has_ocean_physics']

           lat_min      lat_max      lon_min      lon_max   lat_center  \
count  2272.000000  2272.000000  2272.000000  2272.000000  2272.000000   
mean     14.454606    14.679260    69.977855    70.210119    14.567219   
std       4.249297     4.248029     2.601206     2.598397     4.249297   
min       8.000000     8.225225    66.000000    66.226848     8.112613   
25%      10.702703    10.927928    67.703996    67.947424    10.815315   
50%      14.081081    14.306306    69.731732    69.964965    14.193694   
75%      17.909910    18.135135    72.013156    72.249371    18.022523   
max      23.990991    24.000000    75.998813    76.000000    24.103604   

        lon_center  wildlife_count  wildlife_exposure  habitat_se

In [ ]:
import folium

# rebuild geometry for mapping since we saved without it
grid_as_final["geometry"] = grid_as_final.apply(
    lambda r: box(r.lon_min, r.lat_min, r.lon_max, r.lat_max), axis=1
)
grid_as_map = gpd.GeoDataFrame(grid_as_final, geometry="geometry", crs="EPSG:4326")

m = folium.Map(location=[14, 72], zoom_start=6, tiles="CartoDB positron")

def color_for(row):
    if row["habitat_sensitivity"] == 1:
        return "red"
    elif row["wildlife_count"] > 100:
        return "orange"
    elif row["has_ocean_physics"]:
        return "blue"
    else:
        return "gray"

for _, row in grid_as_map.iterrows():
    folium.Rectangle(
        bounds=[[row.lat_min, row.lon_min], [row.lat_max, row.lon_max]],
        color=color_for(row),
        weight=0.5,
        fill=True,
        fill_opacity=0.3,
        popup=f"{row.grid_id}<br>wildlife: {row.wildlife_count}<br>reef: {row.habitat_sensitivity}<br>sst: {row.sst:.2f}°C" if pd.notna(row.sst) else f"{row.grid_id}<br>wildlife: {row.wildlife_count}<br>reef: {row.habitat_sensitivity}<br>no ocean data"
    ).add_to(m)

m.save("/content/drive/MyDrive/AquaStrat/grid_as_visualization.html")
m

In [ ]:
print("Total reef cells:", (grid_as_final["habitat_sensitivity"] == 1).sum())
grid_as_final[grid_as_final["habitat_sensitivity"] == 1][["grid_id","lat_center","lon_center","wildlife_count","habitat_sensitivity"]]

Total reef cells: 28


,grid_id,lat_center,lon_center,wildlife_count,habitat_sensitivity
75,AS_00075,8.337838,72.922795,664.0,1.0
76,AS_00076,8.337838,73.149772,1138.0,1.0
383,AS_00383,9.914414,72.269518,4.0,1.0
427,AS_00427,10.139640,72.273877,282.0,1.0
433,AS_00433,10.139640,73.642723,586.0,1.0
516,AS_00516,10.590090,72.511374,636.0,1.0
517,AS_00517,10.590090,72.739844,580.0,1.0
558,AS_00558,10.815315,72.058936,373.0,1.0
559,AS_00559,10.815315,72.287575,906.0,1.0
560,AS_00560,10.815315,72.516214,151.0,1.0


In [ ]:
import folium
from folium import Element

m = folium.Map(location=[14, 72], zoom_start=6, tiles="CartoDB positron")

def color_for(row):
    if row["habitat_sensitivity"] == 1:
        return "#e63946"   # red — reef
    elif row["wildlife_count"] > 100:
        return "#f4a261"   # orange — high wildlife
    elif row["has_ocean_physics"]:
        return "#457b9d"   # blue — ocean data available
    else:
        return "#adb5bd"   # gray — no ocean data

for _, row in grid_as_map.iterrows():
    tooltip_text = (
        f"<b>{row.grid_id}</b><br>"
        f"Wildlife records: {int(row.wildlife_count)}<br>"
        f"Reef present: {'Yes' if row.habitat_sensitivity == 1 else 'No'}<br>"
        f"SST: {row.sst:.2f}°C<br>Salinity: {row.salinity:.2f} PSU<br>"
        f"Current (u,v): {row.current_u:.3f}, {row.current_v:.3f}"
        if pd.notna(row.sst) else
        f"<b>{row.grid_id}</b><br>Wildlife records: {int(row.wildlife_count)}<br>"
        f"Reef present: {'Yes' if row.habitat_sensitivity == 1 else 'No'}<br>"
        f"Ocean data: not available for this cell"
    )
    folium.Rectangle(
        bounds=[[row.lat_min, row.lon_min], [row.lat_max, row.lon_max]],
        color=color_for(row),
        weight=0.5,
        fill=True,
        fill_opacity=0.45,
        tooltip=folium.Tooltip(tooltip_text, sticky=True)   # shows on HOVER
    ).add_to(m)

# --- Title ---
title_html = """
<div style="position: fixed; top: 10px; left: 50px; z-index: 9999;
            background-color: white; padding: 10px 16px; border-radius: 6px;
            box-shadow: 0 2px 6px rgba(0,0,0,0.3); font-size: 18px; font-weight: bold;">
    AQUA-STRAT — Arabian Sea Grid (2,272 ocean cells, 25km resolution)
</div>
"""
m.get_root().html.add_child(Element(title_html))

# --- Legend ---
legend_html = """
<div style="position: fixed; bottom: 30px; left: 50px; z-index: 9999;
            background-color: white; padding: 12px 16px; border-radius: 6px;
            box-shadow: 0 2px 6px rgba(0,0,0,0.3); font-size: 13px; line-height: 1.8;">
    <b>Grid Cell Legend</b><br>
    <span style="background:#e63946; width:12px; height:12px; display:inline-block; border-radius:2px;"></span> Reef present (habitat sensitivity)<br>
    <span style="background:#f4a261; width:12px; height:12px; display:inline-block; border-radius:2px;"></span> High wildlife density (&gt;100 records)<br>
    <span style="background:#457b9d; width:12px; height:12px; display:inline-block; border-radius:2px;"></span> Ocean physics data available<br>
    <span style="background:#adb5bd; width:12px; height:12px; display:inline-block; border-radius:2px;"></span> No ocean physics coverage<br>
    <hr style="margin:6px 0;">
    <i>Hover over any cell for details</i>
</div>
"""
m.get_root().html.add_child(Element(legend_html))

m.save("/content/drive/MyDrive/AquaStrat/grid_as_visualization.html")
m

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving Wildlife Bay of bengal.zip to Wildlife Bay of bengal.zip


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving ocean_physics.nc to ocean_physics.nc


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving coral_dataset.zip to coral_dataset.zip


In [ ]:
!unzip -q "Wildlife Bay of bengal.zip" -d /content/wildlife_bob
!unzip -q "coral_dataset.zip" -d /content/coral_bob
!find /content/wildlife_bob -maxdepth 4
!find /content/coral_bob -maxdepth 4

/content/wildlife_bob
/content/wildlife_bob/Wildlife Bay of bengal
/content/wildlife_bob/Wildlife Bay of bengal/Occurrence.tsv
/content/wildlife_bob/Wildlife Bay of bengal/citations.html
/content/wildlife_bob/Wildlife Bay of bengal/query.json
/content/coral_bob
/content/coral_bob/Bay of bengal
/content/coral_bob/Bay of bengal/boundary
/content/coral_bob/Bay of bengal/boundary/boundary.geojson
/content/coral_bob/Bay of bengal/Seagrass-2023-2024
/content/coral_bob/Bay of bengal/Seagrass-2023-2024/seagrass_2023_0.tif
/content/coral_bob/Bay of bengal/stats
/content/coral_bob/Bay of bengal/stats/statistics.csv
/content/coral_bob/Bay of bengal/Reef-Extent
/content/coral_bob/Bay of bengal/Reef-Extent/reefextent.geojson
/content/coral_bob/Bay of bengal/1-License-and-Documentation
/content/coral_bob/Bay of bengal/1-License-and-Documentation/User-Guide-Part-1-General-Use.pdf
/content/coral_bob/Bay of bengal/1-License-and-Documentation/Class-Descriptions-Benthic-Maps-v3.pdf
/content/coral_bob/Bay

In [ ]:
grid_bob = build_equal_area_grid("BOB", 6, 22, 80, 93, cell_km=25)
grid_bob_gdf = gpd.GeoDataFrame(
    grid_bob,
    geometry=[box(r.lon_min, r.lat_min, r.lon_max, r.lat_max) for r in grid_bob.itertuples()],
    crs="EPSG:4326"
)

centroids = gpd.GeoDataFrame(geometry=grid_bob_gdf.geometry.centroid, crs="EPSG:4326")
on_land = gpd.sjoin(centroids, world[["geometry"]], how="left", predicate="within")
grid_bob_gdf["is_land"] = ~on_land["index_right"].isna()
grid_bob_gdf = grid_bob_gdf[~grid_bob_gdf["is_land"]].reset_index(drop=True)
print("BOB ocean cells:", grid_bob_gdf.shape)

# ---- WILDLIFE ----
wildlife_bob = pd.read_csv("/content/wildlife_bob/Wildlife Bay of bengal/Occurrence.tsv", sep='\t', low_memory=False)
wildlife_bob = wildlife_bob[['decimalLatitude','decimalLongitude']].dropna()
wildlife_bob_gdf = gpd.GeoDataFrame(wildlife_bob, geometry=gpd.points_from_xy(wildlife_bob.decimalLongitude, wildlife_bob.decimalLatitude), crs="EPSG:4326")
joined = gpd.sjoin(wildlife_bob_gdf, grid_bob_gdf[["grid_id","geometry"]], how="inner", predicate="within")
counts = joined.groupby("grid_id").size().rename("wildlife_count")
grid_bob_gdf = grid_bob_gdf.merge(counts, on="grid_id", how="left").fillna({"wildlife_count": 0})
grid_bob_gdf["wildlife_exposure"] = np.log1p(grid_bob_gdf["wildlife_count"]) / np.log1p(grid_bob_gdf["wildlife_count"]).max()
print("Wildlife:", grid_bob_gdf["wildlife_count"].describe())

# ---- CORAL ----
coral_bob = gpd.read_file("/content/coral_bob/Bay of bengal/Reef-Extent/reefextent.geojson")
joined_coral = gpd.sjoin(grid_bob_gdf[["grid_id","geometry"]], coral_bob, how="inner", predicate="intersects")
grid_bob_gdf["habitat_sensitivity"] = grid_bob_gdf["grid_id"].isin(joined_coral["grid_id"].unique()).astype(float)
print("Reef cells:", grid_bob_gdf["habitat_sensitivity"].sum())

# ---- OCEAN PHYSICS ----
ds_bob = xr.open_dataset("/content/ocean_physics.nc")
print(ds_bob)
ds_bob_mean = ds_bob.mean(dim="time", skipna=True).squeeze("depth")

def get_cell_ocean_stats(row, ds_mean):
    sub = ds_mean.sel(latitude=slice(row.lat_min, row.lat_max), longitude=slice(row.lon_min, row.lon_max))
    if sub.latitude.size == 0 or sub.longitude.size == 0:
        return pd.Series([np.nan]*5)
    return pd.Series([float(sub[v].mean(skipna=True)) for v in ['thetao','so','uo','vo','zos']])

grid_bob_gdf[['sst','salinity','current_u','current_v','ssh']] = grid_bob_gdf.apply(lambda r: get_cell_ocean_stats(r, ds_bob_mean), axis=1)
grid_bob_gdf["has_ocean_physics"] = grid_bob_gdf["sst"].notna()
print("Ocean physics valid cells:", grid_bob_gdf["has_ocean_physics"].sum(), "/", grid_bob_gdf.shape[0])
print(grid_bob_gdf[['sst','salinity','current_u','current_v','ssh']].describe())

# ---- SAVE + DOWNLOAD ----
grid_bob_gdf.drop(columns="geometry").to_csv("/content/grid_bob_final.csv", index=False)
files.download("/content/grid_bob_final.csv")
print("BOB grid complete and downloaded.")

BOB ocean cells: (3349, 10)
Wildlife: count    3349.000000
mean       20.055240
std       104.348137
min         0.000000
25%         0.000000
50%         0.000000
75%         0.000000
max      3262.000000
Name: wildlife_count, dtype: float64
Reef cells: 41.0
<xarray.Dataset> Size: 231MB
Dimensions:    (time: 366, depth: 1, latitude: 175, longitude: 180)
Coordinates:
  * time       (time) datetime64[ns] 3kB 2025-01-01 2025-01-02 ... 2026-01-01
  * depth      (depth) float32 4B 0.494
  * latitude   (latitude) float32 700B 8.0 8.083 8.167 8.25 ... 22.33 22.42 22.5
  * longitude  (longitude) float32 720B 80.0 80.08 80.17 ... 94.75 94.83 94.92
Data variables:
    thetao     (time, depth, latitude, longitude) float32 46MB ...
    so         (time, depth, latitude, longitude) float32 46MB ...
    uo         (time, depth, latitude, longitude) float32 46MB ...
    vo         (time, depth, latitude, longitude) float32 46MB ...
    zos        (time, latitude, longitude) float32 46MB ...
Attribut

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

BOB grid complete and downloaded.


In [ ]:
# Load AS (from Drive) and BOB (already in memory, but reload from disk to be safe)
grid_as_final = pd.read_csv(f"{BASE}/grid_as_final.csv")
grid_bob_final = pd.read_csv("/content/grid_bob_final.csv")

print("AS columns:", grid_as_final.columns.tolist())
print("BOB columns:", grid_bob_final.columns.tolist())

# Align columns in case of any mismatch before concatenating
common_cols = [c for c in grid_as_final.columns if c in grid_bob_final.columns]
grid_master = pd.concat(
    [grid_as_final[common_cols], grid_bob_final[common_cols]],
    ignore_index=True
)

print("\nFinal grid_master shape:", grid_master.shape)
print(grid_master["region"].value_counts())
print("\nTotal reef cells:", grid_master["habitat_sensitivity"].sum())
print("Cells with ocean physics:", grid_master["has_ocean_physics"].sum(), "/", grid_master.shape[0])

# Save + download
grid_master.to_csv("/content/grid_master.csv", index=False)
grid_master.to_csv(f"{BASE}/grid_master.csv", index=False)  # also save to Drive if space allows

from google.colab import files
files.download("/content/grid_master.csv")

grid_master.head()

AS columns: ['grid_id', 'region', 'lat_min', 'lat_max', 'lon_min', 'lon_max', 'lat_center', 'lon_center', 'is_land', 'wildlife_count', 'wildlife_exposure', 'habitat_sensitivity', 'sst', 'salinity', 'current_u', 'current_v', 'ssh', 'has_ocean_physics']
BOB columns: ['grid_id', 'region', 'lat_min', 'lat_max', 'lon_min', 'lon_max', 'lat_center', 'lon_center', 'is_land', 'wildlife_count', 'wildlife_exposure', 'habitat_sensitivity', 'sst', 'salinity', 'current_u', 'current_v', 'ssh', 'has_ocean_physics']

Final grid_master shape: (5621, 18)
region
BOB    3349
AS     2272
Name: count, dtype: int64

Total reef cells: 69.0
Cells with ocean physics: 4581 / 5621


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,grid_id,region,lat_min,lat_max,lon_min,lon_max,lat_center,lon_center,is_land,wildlife_count,wildlife_exposure,habitat_sensitivity,sst,salinity,current_u,current_v,ssh,has_ocean_physics
0,AS_00000,AS,8.0,8.225225,66.000000,66.226848,8.112613,66.113424,False,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,False
1,AS_00001,AS,8.0,8.225225,66.226848,66.453696,8.112613,66.340272,False,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,False
2,AS_00002,AS,8.0,8.225225,66.453696,66.680544,8.112613,66.567120,False,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,False
3,AS_00003,AS,8.0,8.225225,66.680544,66.907392,8.112613,66.793968,False,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,False
4,AS_00004,AS,8.0,8.225225,66.907392,67.134240,8.112613,67.020816,False,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,False


In [ ]:
import folium
from folium import Element

m_final = folium.Map(location=[14, 78], zoom_start=5, tiles="CartoDB positron")

# EXACT same color logic as your original AS map — untouched
def color_for(row):
    if row["habitat_sensitivity"] == 1:
        return "#e63946"   # red — reef
    elif row["wildlife_count"] > 100:
        return "#f4a261"   # orange — high wildlife
    elif row["has_ocean_physics"]:
        return "#457b9d"   # blue — ocean data available
    else:
        return "#adb5bd"   # gray — no ocean data

# EXACT same tooltip logic as your original AS map — untouched
for _, row in grid_master_gdf.iterrows():
    tooltip_text = (
        f"<b>{row.grid_id}</b><br>"
        f"Wildlife records: {int(row.wildlife_count)}<br>"
        f"Reef present: {'Yes' if row.habitat_sensitivity == 1 else 'No'}<br>"
        f"SST: {row.sst:.2f}°C<br>Salinity: {row.salinity:.2f} PSU<br>"
        f"Current (u,v): {row.current_u:.3f}, {row.current_v:.3f}"
        if pd.notna(row.sst) else
        f"<b>{row.grid_id}</b><br>Wildlife records: {int(row.wildlife_count)}<br>"
        f"Reef present: {'Yes' if row.habitat_sensitivity == 1 else 'No'}<br>"
        f"Ocean data: not available for this cell"
    )
    folium.Rectangle(
        bounds=[[row.lat_min, row.lon_min], [row.lat_max, row.lon_max]],
        color=color_for(row),
        weight=0.5,
        fill=True,
        fill_opacity=0.45,
        tooltip=folium.Tooltip(tooltip_text, sticky=True)
    ).add_to(m_final)

title_html = """
<div style="position: fixed; top: 10px; left: 50px; z-index: 9999;
            background-color: white; padding: 10px 16px; border-radius: 6px;
            box-shadow: 0 2px 6px rgba(0,0,0,0.3); font-size: 18px; font-weight: bold;">
    AQUA-STRAT — AS + BOB Grid (5,621 ocean cells, 25km resolution)
</div>
"""
m_final.get_root().html.add_child(Element(title_html))

legend_html = """
<div style="position: fixed; bottom: 30px; left: 50px; z-index: 9999;
            background-color: white; padding: 12px 16px; border-radius: 6px;
            box-shadow: 0 2px 6px rgba(0,0,0,0.3); font-size: 13px; line-height: 1.8;">
    <b>Grid Cell Legend</b><br>
    <span style="background:#e63946; width:12px; height:12px; display:inline-block; border-radius:2px;"></span> Reef present (habitat sensitivity)<br>
    <span style="background:#f4a261; width:12px; height:12px; display:inline-block; border-radius:2px;"></span> High wildlife density (&gt;100 records)<br>
    <span style="background:#457b9d; width:12px; height:12px; display:inline-block; border-radius:2px;"></span> Ocean physics data available<br>
    <span style="background:#adb5bd; width:12px; height:12px; display:inline-block; border-radius:2px;"></span> No ocean physics coverage<br>
    <hr style="margin:6px 0;">
    <i>Hover over any cell for details</i>
</div>
"""
m_final.get_root().html.add_child(Element(legend_html))

m_final.save("/content/grid_master_visualization.html")
files.download("/content/grid_master_visualization.html")
m_final

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving Indian ocean coral dataset.zip to Indian ocean coral dataset.zip


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving Indian ocean region wildlife.zip to Indian ocean region wildlife.zip


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving indian ocean_ocean current.nc to indian ocean_ocean current.nc


In [ ]:
!ls /content/

 coral_bob			      'Indian ocean coral dataset.zip'
 coral_dataset.zip		      'indian ocean_ocean current.nc'
 coral_io			      'Indian ocean region wildlife.zip'
 drive				       ocean_physics.nc
 grid_bob_final.csv		       sample_data
 grid_master.csv		      'Wildlife Bay of bengal (1).zip'
 grid_master_visualization.html       'Wildlife Bay of bengal.zip'
'Indian ocean coral dataset (1).zip'   wildlife_bob


In [ ]:
!unzip -q "Indian ocean region wildlife.zip" -d /content/wildlife_io
!find /content/wildlife_io -maxdepth 4
!find /content/coral_io -maxdepth 4

ds_io = xr.open_dataset("/content/indian ocean_ocean current.nc")
print(ds_io)

/content/wildlife_io
/content/wildlife_io/Indian ocean region
/content/wildlife_io/Indian ocean region/Occurrence.tsv
/content/wildlife_io/Indian ocean region/citations.html
/content/wildlife_io/Indian ocean region/query.json
/content/coral_io
/content/coral_io/Indian ocean
/content/coral_io/Indian ocean/boundary
/content/coral_io/Indian ocean/boundary/boundary.geojson
/content/coral_io/Indian ocean/Seagrass-2023-2024
/content/coral_io/Indian ocean/Seagrass-2023-2024/seagrass_2023_0.tif
/content/coral_io/Indian ocean/stats
/content/coral_io/Indian ocean/stats/statistics.csv
/content/coral_io/Indian ocean/Reef-Extent
/content/coral_io/Indian ocean/Reef-Extent/reefextent.geojson
/content/coral_io/Indian ocean/1-License-and-Documentation
/content/coral_io/Indian ocean/1-License-and-Documentation/User-Guide-Part-1-General-Use.pdf
/content/coral_io/Indian ocean/1-License-and-Documentation/Class-Descriptions-Benthic-Maps-v3.pdf
/content/coral_io/Indian ocean/1-License-and-Documentation/Metho

In [ ]:
grid_io = build_equal_area_grid("IO", -20, 25, 40, 100, cell_km=50)
grid_io_gdf = gpd.GeoDataFrame(
    grid_io,
    geometry=[box(r.lon_min, r.lat_min, r.lon_max, r.lat_max) for r in grid_io.itertuples()],
    crs="EPSG:4326"
)
print("Total IO grid cells (pre-mask):", grid_io_gdf.shape)

centroids = gpd.GeoDataFrame(geometry=grid_io_gdf.geometry.centroid, crs="EPSG:4326")
on_land = gpd.sjoin(centroids, world[["geometry"]], how="left", predicate="within")
grid_io_gdf["is_land"] = ~on_land["index_right"].isna()
grid_io_gdf = grid_io_gdf[~grid_io_gdf["is_land"]].reset_index(drop=True)
print("IO ocean cells:", grid_io_gdf.shape)

# ---- WILDLIFE ----
wildlife_io = pd.read_csv("/content/wildlife_io/Indian ocean region/Occurrence.tsv", sep='\t', low_memory=False)
wildlife_io = wildlife_io[['decimalLatitude','decimalLongitude']].dropna()
wildlife_io_gdf = gpd.GeoDataFrame(wildlife_io, geometry=gpd.points_from_xy(wildlife_io.decimalLongitude, wildlife_io.decimalLatitude), crs="EPSG:4326")
joined = gpd.sjoin(wildlife_io_gdf, grid_io_gdf[["grid_id","geometry"]], how="inner", predicate="within")
counts = joined.groupby("grid_id").size().rename("wildlife_count")
grid_io_gdf = grid_io_gdf.merge(counts, on="grid_id", how="left").fillna({"wildlife_count": 0})
grid_io_gdf["wildlife_exposure"] = np.log1p(grid_io_gdf["wildlife_count"]) / np.log1p(grid_io_gdf["wildlife_count"]).max()
print("Wildlife:", grid_io_gdf["wildlife_count"].describe())

# ---- CORAL ----
coral_io = gpd.read_file("/content/coral_io/Indian ocean/Reef-Extent/reefextent.geojson")
joined_coral = gpd.sjoin(grid_io_gdf[["grid_id","geometry"]], coral_io, how="inner", predicate="intersects")
grid_io_gdf["habitat_sensitivity"] = grid_io_gdf["grid_id"].isin(joined_coral["grid_id"].unique()).astype(float)
print("Reef cells:", grid_io_gdf["habitat_sensitivity"].sum())

# ---- OCEAN PHYSICS ----
ds_io_mean = ds_io.mean(dim="time", skipna=True).squeeze("depth")

def get_cell_ocean_stats(row, ds_mean):
    sub = ds_mean.sel(latitude=slice(row.lat_min, row.lat_max), longitude=slice(row.lon_min, row.lon_max))
    if sub.latitude.size == 0 or sub.longitude.size == 0:
        return pd.Series([np.nan]*5)
    return pd.Series([float(sub[v].mean(skipna=True)) for v in ['thetao','so','uo','vo','zos']])

grid_io_gdf[['sst','salinity','current_u','current_v','ssh']] = grid_io_gdf.apply(lambda r: get_cell_ocean_stats(r, ds_io_mean), axis=1)
grid_io_gdf["has_ocean_physics"] = grid_io_gdf["sst"].notna()
print("Ocean physics valid cells:", grid_io_gdf["has_ocean_physics"].sum(), "/", grid_io_gdf.shape[0])
print(grid_io_gdf[['sst','salinity','current_u','current_v','ssh']].describe())

# ---- SAVE + DOWNLOAD ----
grid_io_gdf.drop(columns="geometry").to_csv("/content/grid_io_final.csv", index=False)
files.download("/content/grid_io_final.csv")
print("IO grid complete and downloaded.")

Total IO grid cells (pre-mask): (13055, 9)
IO ocean cells: (10355, 10)
Wildlife: count    10355.000000
mean         5.835249
std        136.978821
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max       8624.000000
Name: wildlife_count, dtype: float64
Reef cells: 35.0
Ocean physics valid cells: 1802 / 10355
               sst     salinity    current_u    current_v          ssh
count  1802.000000  1802.000000  1802.000000  1802.000000  1802.000000
mean     29.497937    34.400471     0.070736    -0.013048     0.539604
std       0.262386     0.561497     0.113395     0.042644     0.061706
min      28.029327    32.253002    -0.269279    -0.325529     0.357400
25%      29.318383    34.151390    -0.012059    -0.031434     0.493082
50%      29.510485    34.538601     0.049351    -0.011494     0.534856
75%      29.695671    34.806084     0.148319     0.008211     0.588290
max      30.018461    35.462872     0.406805     0.193530     0.692035


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

IO grid complete and downloaded.


In [ ]:
print(ds_io.latitude.values[0], ds_io.latitude.values[-1])

-5.0 8.0


In [ ]:
import pandas as pd, numpy as np, geopandas as gpd, folium
from folium import Element
from shapely.geometry import box
from google.colab import drive, files
import warnings
warnings.filterwarnings("ignore")

drive.mount('/content/drive')
BASE = "/content/drive/MyDrive/AquaStrat"

Mounted at /content/drive


In [ ]:
drive.flush_and_unmount()

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
grid_as_final = pd.read_csv(f"{BASE}/grid_as_final.csv")
print(grid_as_final.shape)

(2272, 18)


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving grid_io_final.csv to grid_io_final.csv
Saving grid_bob_final.csv to grid_bob_final.csv


In [ ]:
!ls /content/*.csv

/content/grid_bob_final.csv  /content/grid_io_final.csv


In [ ]:
import pandas as pd, numpy as np, geopandas as gpd, folium
from folium import Element
from shapely.geometry import box
from google.colab import drive, files
import warnings
warnings.filterwarnings("ignore")

drive.mount('/content/drive')
BASE = "/content/drive/MyDrive/AquaStrat"

Mounted at /content/drive


In [ ]:
grid_as_final = pd.read_csv(f"{BASE}/grid_as_final.csv")
print(grid_as_final.shape)

(2272, 18)


In [ ]:
!ls /content/*.csv

ls: cannot access '/content/*.csv': No such file or directory


In [ ]:
import pandas as pd, numpy as np, geopandas as gpd, folium
from folium import Element
from shapely.geometry import box
from google.colab import drive, files
import warnings
warnings.filterwarnings("ignore")

drive.mount('/content/drive')
BASE = "/content/drive/MyDrive/AquaStrat"

# reload AS from Drive
grid_as_final = pd.read_csv(f"{BASE}/grid_as_final.csv")
print("AS:", grid_as_final.shape)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
AS: (2272, 18)


In [ ]:
uploaded = files.upload()

Saving grid_io_final.csv to grid_io_final.csv
Saving grid_bob_final.csv to grid_bob_final.csv


In [ ]:
grid_bob_final = pd.read_csv("/content/grid_bob_final.csv")
grid_io_final = pd.read_csv("/content/grid_io_final.csv")

common_cols = [c for c in grid_as_final.columns if c in grid_bob_final.columns and c in grid_io_final.columns]
grid_master_full = pd.concat(
    [grid_as_final[common_cols], grid_bob_final[common_cols], grid_io_final[common_cols]],
    ignore_index=True
)

print("Final shape:", grid_master_full.shape)
grid_master_full.to_csv(f"{BASE}/grid_master_full.csv", index=False)
grid_master_full.to_csv("/content/grid_master_full.csv", index=False)
files.download("/content/grid_master_full.csv")

Final shape: (15976, 18)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
grid_master_full = pd.read_csv(f"{BASE}/grid_master_full.csv")
print(grid_master_full.shape)

(15976, 18)


In [ ]:
print(grid_master_full["region"].value_counts())
print("Total reef cells:", grid_master_full["habitat_sensitivity"].sum())
print("Cells with ocean physics:", grid_master_full["has_ocean_physics"].sum(), "/", grid_master_full.shape[0])
grid_master_full.groupby("region")[["wildlife_count","habitat_sensitivity","has_ocean_physics"]].sum()

region
IO     10355
BOB     3349
AS      2272
Name: count, dtype: int64
Total reef cells: 104.0
Cells with ocean physics: 6383 / 15976


,wildlife_count,habitat_sensitivity,has_ocean_physics
region,,,
AS,84210.0,28.0,1673
BOB,67165.0,41.0,2908
IO,60424.0,35.0,1802


In [ ]:
# rebuild geometry
grid_master_full["geometry"] = grid_master_full.apply(lambda r: box(r.lon_min, r.lat_min, r.lon_max, r.lat_max), axis=1)

# AS + BOB: show fully. IO: only cells with actual data (wildlife, reef, or ocean physics)
as_bob = grid_master_full[grid_master_full["region"].isin(["AS", "BOB"])]
io_filtered = grid_master_full[
    (grid_master_full["region"] == "IO") &
    ((grid_master_full["wildlife_count"] > 0) | (grid_master_full["habitat_sensitivity"] == 1) | (grid_master_full["has_ocean_physics"] == True))
]

map_data = pd.concat([as_bob, io_filtered], ignore_index=True)
map_gdf = gpd.GeoDataFrame(map_data, geometry="geometry", crs="EPSG:4326")
print("Cells shown on map:", map_gdf.shape[0], "( IO filtered from", (grid_master_full['region']=='IO').sum(), "down to", io_filtered.shape[0], ")")

m_all = folium.Map(location=[5, 75], zoom_start=4, tiles="CartoDB positron")

region_colors = {"AS": "#457b9d", "BOB": "#2a9d8f", "IO": "#8d99ae"}

def color_for(row):
    if row["habitat_sensitivity"] == 1:
        return "#e63946"   # red — reef
    elif row["wildlife_count"] > 100:
        return "#f4a261"   # orange — high wildlife
    else:
        return region_colors[row["region"]]

for _, row in map_gdf.iterrows():
    tooltip_text = (
        f"<b>{row.grid_id}</b> ({row.region})<br>"
        f"Wildlife records: {int(row.wildlife_count)}<br>"
        f"Reef present: {'Yes' if row.habitat_sensitivity == 1 else 'No'}<br>"
        + (f"SST: {row.sst:.2f}°C<br>Salinity: {row.salinity:.2f} PSU<br>Current (u,v): {row.current_u:.3f}, {row.current_v:.3f}"
           if pd.notna(row.sst) else "Ocean physics: not available")
    )
    folium.Rectangle(
        bounds=[[row.lat_min, row.lon_min], [row.lat_max, row.lon_max]],
        color=color_for(row), weight=0.4, fill=True,
        fill_opacity=0.55 if row["habitat_sensitivity"]==1 or row["wildlife_count"]>100 else 0.25,
        tooltip=folium.Tooltip(tooltip_text, sticky=True)
    ).add_to(m_all)

title_html = """<div style="position: fixed; top: 10px; left: 50px; z-index: 9999; background:white; padding:10px 16px; border-radius:6px; box-shadow:0 2px 6px rgba(0,0,0,0.3); font-size:16px; font-weight:bold;">AQUA-STRAT — AS + BOB + IO Grid (15,976 cells total)</div>"""
legend_html = """<div style="position: fixed; bottom: 30px; left: 50px; z-index: 9999; background:white; padding:12px 16px; border-radius:6px; box-shadow:0 2px 6px rgba(0,0,0,0.3); font-size:13px; line-height:1.8;">
<b>Legend</b><br>
<span style="background:#457b9d;width:12px;height:12px;display:inline-block;"></span> Arabian Sea<br>
<span style="background:#2a9d8f;width:12px;height:12px;display:inline-block;"></span> Bay of Bengal<br>
<span style="background:#8d99ae;width:12px;height:12px;display:inline-block;"></span> Indian Ocean (data cells only)<br>
<span style="background:#f4a261;width:12px;height:12px;display:inline-block;"></span> High wildlife (&gt;100 records)<br>
<span style="background:#e63946;width:12px;height:12px;display:inline-block;"></span> Reef present<br>
<i>Hover for cell details</i></div>"""
m_all.get_root().html.add_child(Element(title_html))
m_all.get_root().html.add_child(Element(legend_html))

m_all.save(f"{BASE}/grid_master_full_visualization.html")
m_all.save("/content/grid_master_full_visualization.html")
files.download("/content/grid_master_full_visualization.html")
m_all

Cells shown on map: 7423 ( IO filtered from 10355 down to 1802 )


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>